# voice

> Speech to text on this machine, for think-aloud answers, and a natural voice that reads the guide aloud.

This module turns a spoken answer into text. With the local engines, the audio does not go off the machine.
The web app records while you talk. It cuts the audio into segments at your pauses and sends each segment here.
This module transcribes the segments one at a time, in order. The text goes into the think-aloud box while you continue to talk.

The module also speaks. `speak` turns one line of the screen guide into WAV audio, with the start time of each word, so the board can light each word as it is said.

In [ ]:
#| default_exp voice

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['VISHALAKSHI_VAULT'] = os.environ['ABHYASA_VAULT'] = os.path.join(tempfile.mkdtemp(), 'vault.db')   # a test that opens the default vault gets a temporary one
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # and the settings and keys go to a temporary folder
os.environ['RAMABANA_CONFIG'] = os.path.join(tempfile.mkdtemp(), 'config.toml')   # and ramabana reads no real config
for k in ('ABHYASA_STT', 'OPENAI_API_KEY'): os.environ.pop(k, None)

These are the imports of the module.

In [ ]:
#| export
import os, re, gc, json, time, uuid, inspect, tempfile, threading, wave
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from fastcore.utils import *
from abhyasa.prompts import register, prompt

## Engines

There are three engines. `gemma` and `mlx` run on this machine. `openai` sends the audio to OpenAI.

- `gemma` runs a Gemma 4 model on LiteRT through the model layer. E4B is the default. E2B is faster.
- `mlx` runs any speech to text model that mlx-audio can load: Whisper, Parakeet, Qwen3-ASR, Voxtral, Canary, Moonshine and more.
  Install it with `pip install abhyasa[whisper]`.
- `openai` uses the OpenAI transcription API. It is off until you set a key. It sends the audio off the machine.

Each engine gets a hint: the name of the concept and its terms. Gemma reads it in its prompt. Whisper reads it as its initial prompt,
and Qwen3-ASR as hot words. OpenAI reads it as its prompt.

The table gives the numbers on this machine (an M-series Mac), with each model warm.
"Clip A" is the English test clip of the model layer (`rishi/nbs/speech.wav`).
"Clip B" is a clip in a cloned voice with Nyāya terms in IAST and technical terms.
"Terms" is the fraction of the 9 key terms in clip B that the engine wrote correctly.
"5 s" is the time for one 5-second segment.

| model | load | 5 s | clip A | clip B, no hint / hint | terms, no hint / hint | Sanskrit CER | memory |
|---|---|---|---|---|---|---|---|
| `litert-community/gemma-4-E4B-it-litert-lm` | 5.0 s | 0.69 s | 9% | 0% / 0% | 9 / 9 | 12% | 1.5 GB |
| `litert-community/gemma-4-E2B-it-litert-lm` | 2.6 s | 0.38 s | 14% | 6% / 2% | 7 / 8 | 13% | 1.3 GB |
| `mlx-community/whisper-large-v3-turbo-asr-fp16` | 4.8 s | 0.68 s | 6% | 4% / 0% | 8 / 9 | 10% | 2.3 GB |
| `mlx-community/parakeet-tdt-0.6b-v3` | 4.9 s | 0.15 s | 6% | 4% / 4% | 7 / 7 | Latin script | 3.2 GB |
| `mlx-community/Qwen3-ASR-0.6B-8bit` | 3.8 s | 0.14 s | 6% | 13% / 0% | 5 / 9 | 16% | 2.0 GB |
| `mlx-community/Qwen3-ASR-1.7B-8bit` | 3.6 s | 0.25 s | 6% | 6% / 0% | 7 / 9 | 14% | 3.6 GB |
| `mlx-community/Voxtral-Mini-4B-Realtime-2602-4bit` | 3.7 s | 2.21 s | 6% | 6% / 6% | 7 / 7 | 22% | 5.5 GB |
| `UsefulSensors/moonshine-base` | 3.9 s | 0.07 s | 100% | 8% / 8% | 6 / 6 | Latin script | 0.8 GB |
| `Mediform/canary-1b-v2-mlx-q8` | 4.2 s | 0.08 s | 6% | 4% / 4% | 8 / 8 | Latin script | 1.4 GB |
| Su-śrotā (NeMo, CPU, in vachaka) | 46 s | 0.3 s for 9 s | Devanagari only | Devanagari only | — | 0% | 1.7 GB |

- "load" is a cold start of a cached model in a new process, with the Python imports. The first download takes longer.
- "memory" is the MLX peak for the `mlx` models, and the growth of the process for Gemma on LiteRT.
- "Sanskrit CER" is the character error rate on a 9-second Nyāya sūtra (from vachaka), after the vachaka normalization.
  vachaka chose that clip with Su-śrotā, so the 0% of Su-śrotā is not a fair result.
- Moonshine gave no text for the 24-second clip A. It is good only for short segments.
- Su-śrotā writes only Devanagari, and it needs NeMo and torch on Python 3.12 or older. It is not an engine here.
  For Sanskrit speech, use Gemma or Whisper with the hint, or vachaka.

Gemma 4 E4B stays the default: it has no errors on clip B, it writes all 9 terms without a hint, and one 5-second segment takes 0.7 s.
For more speed, use Parakeet (0.15 s) or Qwen3-ASR 0.6B with the hint (0.14 s, all 9 terms).

`STT_SP` is the system prompt for the Gemma engine. It is the prompt `stt`, so you can change it on the Settings page. The other constants are the default models.
`ALIASES` keeps the old engine names: `whisper` and `parakeet` are the `mlx` engine with a fixed model.

In [ ]:
#| export
STT_SP = register('stt', 'Speech to text', 'How the Gemma speech engine writes down what you say. A change works when the model loads again.',
                  'You transcribe speech exactly. Keep technical terms, Sanskrit words (in IAST) and maths as spoken. Reply with the transcript only.')
DFLT_GEMMA = 'litert-community/gemma-4-E4B-it-litert-lm'
GEMMA_E2B = 'litert-community/gemma-4-E2B-it-litert-lm'
DFLT_WHISPER = 'mlx-community/whisper-large-v3-turbo-asr-fp16'
DFLT_PARAKEET = 'mlx-community/parakeet-tdt-0.6b-v3'
DFLT_OPENAI = 'gpt-4o-mini-transcribe'
DFLT_MODELS = dict(gemma=DFLT_GEMMA, mlx=DFLT_WHISPER, openai=DFLT_OPENAI)
ALIASES = dict(whisper=('mlx', DFLT_WHISPER), parakeet=('mlx', DFLT_PARAKEET))

## Settings

The settings are in `settings.json` in `cfg_dir()`. The `settings` module keeps the table of values and the file; this module uses its `stt` section.
The web app changes the settings, and the next segment uses the change. You do not have to restart.

In [ ]:
#| export
from abhyasa.settings import cfg_dir, DFLT_SETTINGS, settings, save_settings, update_settings, val

Without a file, you get the defaults. A change to one key keeps the other keys.

In [ ]:
test_eq({k: settings()['stt'][k] for k in ('engine', 'model', 'segment')}, dict(engine='gemma', model='', segment=True))
update_settings('stt', segment=False)
test_eq(val('stt.segment'), False)
update_settings('stt', segment=True)

`current` gives the engine and model in use. `$ABHYASA_STT` has priority over the settings. It can be an engine (`gemma`),
an old name (`whisper`), or an engine and a model (`mlx:mlx-community/Qwen3-ASR-0.6B-8bit`).

In [ ]:
#| export
def resolve(engine:str, model:str='') -> tuple:
    "The engine and model for a name: aliases become `mlx`, `engine:model` splits, an empty model gets the default."
    if ':' in engine: engine, model = engine.split(':', 1)
    if engine in ALIASES: engine, model = ALIASES[engine][0], model or ALIASES[engine][1]
    if engine not in DFLT_MODELS: raise ValueError(f'unknown speech engine {engine!r}')
    if engine == 'gemma': model = model or os.getenv('ABHYASA_GEMMA') or DFLT_GEMMA
    return engine, model or DFLT_MODELS[engine]

def current() -> tuple:
    "The engine and model in use: `$ABHYASA_STT`, else the settings."
    if (e := os.getenv('ABHYASA_STT')): return resolve(e)
    s = settings()['stt']
    return resolve(s['engine'], s['model'])

The settings give Gemma E4B by default. The environment variable has priority.

In [ ]:
test_eq(current(), ('gemma', DFLT_GEMMA))
test_eq(resolve('parakeet'), ('mlx', DFLT_PARAKEET))
test_eq(resolve('mlx:mlx-community/Qwen3-ASR-0.6B-8bit'), ('mlx', 'mlx-community/Qwen3-ASR-0.6B-8bit'))
os.environ['ABHYASA_STT'] = 'whisper'
test_eq(current(), ('mlx', DFLT_WHISPER))
del os.environ['ABHYASA_STT']

## One model in memory

`resident` keeps one model loaded in each slot: `stt` for speech to text, `tts` for the speaking voice. When the app asks for a different model
in a slot, it frees the old model of that slot first. Two large models for the same job in memory at the same time can make the Mac slow.

In [ ]:
#| export
_res, _lock = AttrDict(key=None, obj=None, close=None), threading.RLock()
_slots = dict(stt=(_res, _lock), tts=(AttrDict(key=None, obj=None, close=None), threading.RLock()))   # each slot has its own lock, so a load of one does not wait for the other

def unload(slot:str='stt'):
    "Free the loaded model of `slot`: `stt` (speech to text) or `tts` (the speaking voice)."
    r, lock = _slots[slot]
    with lock:
        if r.close and r.obj is not None: r.close(r.obj)
        r.update(key=None, obj=None, close=None)
        gc.collect()
        try:
            import mlx.core as mx; mx.clear_cache()
        except ImportError: pass

def resident(key, load, close=None, slot:str='stt'):
    "The loaded model of `slot` for `key`: when `key` is new, free the old model of the slot, then `load()` the new one."
    r, lock = _slots[slot]
    with lock:
        if r.key != key:
            unload(slot)
            r.obj = load()
            r.key, r.close = key, close
        return r.obj

A new key frees the old model. The same key does not load again. A model in the `tts` slot does not free the model in the `stt` slot.

In [ ]:
freed = []
resident('a', lambda: 'A', freed.append)
test_eq(resident('a', lambda: 'other'), 'A')
test_eq(resident('b', lambda: 'B'), 'B')
test_eq(freed, ['A'])
test_eq(resident('k', lambda: 'K', slot='tts'), 'K')
test_eq((resident('b', lambda: 'other'), freed), ('B', ['A']))
unload(); unload('tts')

Each engine is a function that takes a WAV path, a hint and a model, and returns text.
Gemma keeps one LiteRT engine loaded, and makes a new conversation for each segment. Then one segment does not change the next.

In [ ]:
#| export
def _gemma_engine(model):
    from rishi.litert import LitertChat
    return resident(('gemma', model), lambda: LitertChat.create_engine(model).__enter__(), lambda e: e.__exit__(None, None, None))

def _gemma(path, hint='', model=DFLT_GEMMA):
    from rishi import Chat, resp_text
    chat = Chat(model, engine=_gemma_engine(model), sp=prompt('stt'))
    try: return resp_text(chat([f'Transcribe this clip.{" Words it may use: " + hint if hint else ""}', Path(path)]))
    finally: chat.close()

The `mlx` engine gives the hint to a model that accepts one: Whisper as `initial_prompt`, Qwen3-ASR as `hotwords`.

In [ ]:
#| export
def _hint_kw(m, hint:str) -> dict:
    "The `generate` argument that carries the hint, for models that accept one."
    ps = inspect.signature(m.generate).parameters
    if not hint: return {}
    if 'initial_prompt' in ps: return dict(initial_prompt=hint)
    if 'hotwords' in ps: return dict(hotwords=[o.strip() for o in hint.split(',') if o.strip()])
    return {}

def _mlx(path, hint='', model=DFLT_WHISPER):
    from mlx_audio.stt.utils import load
    m = resident(('mlx', model), lambda: load(model))
    return m.generate(str(path), **_hint_kw(m, hint)).text

A Whisper-like model gets `initial_prompt`. A Qwen3-like model gets a list of hot words. A model without either gets nothing.

In [ ]:
class _W:
    def generate(self, audio, initial_prompt=None): pass
class _Q:
    def generate(self, audio, hotwords=None): pass
test_eq(_hint_kw(_W(), 'Nyāya, hetu'), dict(initial_prompt='Nyāya, hetu'))
test_eq(_hint_kw(_Q(), 'Nyāya, hetu'), dict(hotwords=['Nyāya', 'hetu']))
test_eq(_hint_kw(_W(), ''), {})

## OpenAI

The `openai` engine posts the WAV to the OpenAI transcription API, with the hint as its `prompt`.
`http_post` is `httpx.post` unless a test puts a fake in its place.

In [ ]:
#| export
OPENAI_URL = 'https://api.openai.com/v1/audio/transcriptions'
http_post = None

def _openai(path, hint='', model=DFLT_OPENAI):
    key, _ = openai_key()
    if not key: raise RuntimeError('There is no OpenAI key. Set one on the Settings page.')
    post = http_post or __import__('httpx').post
    data = dict(model=model, response_format='json', **(dict(prompt=hint) if hint else {}))
    r = post(OPENAI_URL, headers={'Authorization': f'Bearer {key}'}, data=data,
             files=dict(file=('clip.wav', Path(path).read_bytes(), 'audio/wav')), timeout=60)
    r.raise_for_status()
    return r.json()['text']

ENGINES = dict(gemma=_gemma, mlx=_mlx, openai=_openai)

The key comes from the first of these places that has `OPENAI_API_KEY`: the environment, the abhyasa `keys.env`, then the ramabana `keys.env`. ramabana says where its file is (`$RAMABANA_CONFIG`, else `~/.config/ramabana/keys.env`).
abhyasa reads both files with ramabana's `load_keys`. It refuses a file that other users can read, and it runs `!command` values. abhyasa only reads the ramabana file. `set_openai_key` writes the abhyasa file with `save_key` (atomic, mode 0600).
ramabana is an optional extra. Without it, abhyasa reads and writes plain `NAME=value` lines.
`masked` shows only the last 4 characters.

In [ ]:
#| export
from fastcore.xtras import atomic_save
try: from ramabana.setup import keys_path, load_keys, save_key
except ImportError:   # ramabana is an optional extra: the same three names, for plain `NAME=value` lines
    def keys_path(): return Path('~/.config/ramabana/keys.env').expanduser()
    def load_keys(path=None):
        p, got = Path(path or keys_path()), []
        for l in (p.read_text().splitlines() if p.exists() and not p.stat().st_mode & 0o077 else []):
            k, _, v = l.strip().removeprefix('export ').partition('=')
            if k.strip() not in os.environ and v: os.environ[k.strip()] = v.strip().strip('"\''); got.append(k.strip())
        return got
    def save_key(name, value, path=None):
        p = Path(path or keys_path())
        keep = [l for l in (p.read_text().splitlines() if p.exists() else []) if l.strip().removeprefix('export ').partition('=')[0].strip() != name]
        with atomic_save(p, 'w') as f: f.write(''.join(l + '\n' for l in keep + [f'{name}={value}']))
        p.chmod(0o600); os.environ[name] = value; return p

def _file_key(p, name='OPENAI_API_KEY'):
    "`name` from the key file `p`, read by `load_keys`; no key stays in the environment."
    got = load_keys(p)
    k = os.environ.get(name, '') if name in got else ''
    for n in got: os.environ.pop(n, None)
    return k

def openai_key() -> tuple:
    "The OpenAI key and where it is from; `('', '')` when there is none."
    if (k := os.getenv('OPENAI_API_KEY')): return k, 'environment'
    for p in (cfg_dir()/'keys.env', keys_path()):
        if (k := _file_key(p)): return k, str(p)
    return '', ''

def set_openai_key(key:str=''):
    "Save `key` in abhyasa's `keys.env` (mode 0600); an empty key clears it. A key with other than letters, digits, `-` and `_` raises `ValueError`."
    if key.strip() and not re.fullmatch(r'[A-Za-z0-9_-]{8,300}', key.strip()): raise ValueError('An API key has only letters, digits, - and _.')
    old = os.environ.get('OPENAI_API_KEY')
    save_key('OPENAI_API_KEY', key.strip(), cfg_dir()/'keys.env')
    if old is None: os.environ.pop('OPENAI_API_KEY', None)   # `save_key` also sets the environment: only a file holds this key
    else: os.environ['OPENAI_API_KEY'] = old

def masked(key:str) -> str: return f'…{key[-4:]}' if key else ''

A key in the abhyasa file comes back with its source, and the file has mode 0600. When the abhyasa file has no key, abhyasa reads the ramabana file. This test points ramabana to a temp folder.

In [ ]:
os.environ['RAMABANA_CONFIG'] = str(Path(tempfile.mkdtemp())/'config.toml')
test_eq(openai_key(), ('', ''))
keys_path().write_text('export OPENAI_API_KEY="sk-ramabana-0000"\n'); keys_path().chmod(0o600)
test_eq(openai_key(), ('sk-ramabana-0000', str(keys_path())))
test_eq(os.getenv('OPENAI_API_KEY'), None)
set_openai_key('sk-test-1234')
test_eq(openai_key(), ('sk-test-1234', str(cfg_dir()/'keys.env')))
test_eq(oct((cfg_dir()/'keys.env').stat().st_mode & 0o777), '0o600')
test_eq(masked('sk-test-1234'), '…1234')

The engine sends the model, the hint and the WAV, with the key in the header. This test uses a fake HTTP client.

In [ ]:
class _R:
    def raise_for_status(self): pass
    def json(self): return dict(text='hetu and sādhya')
sent = {}
def fake_post(url, **kw): sent.update(url=url, **kw); return _R()
http_post = fake_post
p = Path(tempfile.mkdtemp())/'a.wav'; p.write_bytes(b'RIFF')
test_eq(_openai(p, 'hetu'), 'hetu and sādhya')
test_eq(sent['data'], dict(model=DFLT_OPENAI, response_format='json', prompt='hetu'))
test_eq(sent['headers']['Authorization'], 'Bearer sk-test-1234')
set_openai_key('')
keys_path().unlink()
test_fail(lambda: _openai(p), contains='no OpenAI key')
http_post = None

A key with a newline or a space can add a second line to the file. Thus `set_openai_key` refuses it.

In [ ]:
test_fail(lambda: set_openai_key('sk-a\nOTHER=1'), contains='only letters')
test_fail(lambda: set_openai_key('sk a b c d e'), contains='only letters')

## Transcribe

`transcribe` accepts a path or bytes. It writes bytes to a temporary file and removes the file after use.
Without an engine, it uses `current()`, so a change on the Settings page applies to the next segment.

In [ ]:
#| export
def transcribe(audio, engine:str=None, hint:str='', model:str=None) -> str:
    "Text for a WAV file or WAV bytes. `hint` lists words that the speaker can use, such as the name of the concept."
    engine, model = resolve(engine, model or '') if engine else current()
    eng = partial(ENGINES[engine], model=model)
    if isinstance(audio, (bytes, bytearray)):
        with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as f: f.write(audio)
        try: return eng(f.name, hint).strip()
        finally: os.unlink(f.name)
    return eng(audio, hint).strip()

`wav_seconds` gives the length of a clip in seconds. `silence` makes a clip of silence. The app uses this clip to warm a model.

In [ ]:
#| export
def wav_seconds(audio:bytes) -> float:
    "The length of WAV bytes, in seconds; 0 when they are not WAV."
    import io
    try:
        with wave.open(io.BytesIO(audio)) as w: return w.getnframes() / w.getframerate()
    except Exception: return 0.

def silence(secs:float=.5) -> bytes:
    "WAV bytes of silence: 16 kHz, mono, 16 bit."
    import io
    b = io.BytesIO()
    with wave.open(b, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(16000); w.writeframes(b'\0\0' * int(16000 * secs))
    return b.getvalue()

Half a second of silence is 0.5 seconds. Bytes that are not WAV give 0.

In [ ]:
test_eq(wav_seconds(silence()), .5)
test_eq(wav_seconds(b'nope'), 0.)

## Models

`KNOWN` lists the models that we measured (see the table above). The Settings page shows them first.
`cached` tells if a model is in the Hugging Face cache. It does not use the network.

In [ ]:
#| export
KNOWN = [
    dict(engine='gemma', model=DFLT_GEMMA, bench={'seg5': 0.69, 'wer_a': 0.086, 'wer_b': 0.0, 'terms': 9, 'mem': 1.5}, label='Gemma 4 E4B (LiteRT)', note='the default; good with IAST in the hint'),
    dict(engine='gemma', model=GEMMA_E2B, bench={'seg5': 0.38, 'wer_a': 0.143, 'wer_b': 0.021, 'terms': 8, 'mem': 1.3}, label='Gemma 4 E2B (LiteRT)', note='faster than E4B'),
    dict(engine='mlx', model=DFLT_WHISPER, bench={'seg5': 0.68, 'wer_a': 0.057, 'wer_b': 0.0, 'terms': 9, 'mem': 2.3}, label='Whisper large-v3-turbo', note='reads the hint as its initial prompt'),
    dict(engine='mlx', model=DFLT_PARAKEET, bench={'seg5': 0.15, 'wer_a': 0.057, 'wer_b': 0.043, 'terms': 7, 'mem': 3.2}, label='Parakeet TDT 0.6B v3', note='the fastest; European languages only'),
    dict(engine='mlx', model='mlx-community/Qwen3-ASR-0.6B-8bit', bench={'seg5': 0.14, 'wer_a': 0.057, 'wer_b': 0.0, 'terms': 9, 'mem': 2.0}, label='Qwen3-ASR 0.6B (8 bit)', note='reads the hint as hot words'),
    dict(engine='mlx', model='mlx-community/Qwen3-ASR-1.7B-8bit', bench={'seg5': 0.25, 'wer_a': 0.057, 'wer_b': 0.0, 'terms': 9, 'mem': 3.6}, label='Qwen3-ASR 1.7B (8 bit)', note='reads the hint as hot words'),
    dict(engine='mlx', model='mlx-community/Voxtral-Mini-4B-Realtime-2602-4bit', bench={'seg5': 2.21, 'wer_a': 0.057, 'wer_b': 0.064, 'terms': 7, 'mem': 5.5}, label='Voxtral Mini 4B Realtime (4 bit)', note='made for live speech'),
    dict(engine='mlx', model='UsefulSensors/moonshine-base', bench={'seg5': 0.07, 'wer_a': 1.0, 'wer_b': 0.085, 'terms': 6, 'mem': 0.8}, label='Moonshine base', note='small; English only; short segments only'),
    dict(engine='mlx', model='Mediform/canary-1b-v2-mlx-q8', bench={'seg5': 0.08, 'wer_a': 0.057, 'wer_b': 0.043, 'terms': 8, 'mem': 1.4}, label='Canary 1B v2 (8 bit)', note='European languages'),
    dict(engine='openai', model=DFLT_OPENAI, label='OpenAI gpt-4o-mini-transcribe', note='sends audio off this machine; needs a key'),
    dict(engine='openai', model='gpt-4o-transcribe', label='OpenAI gpt-4o-transcribe', note='sends audio off this machine; needs a key'),
]

def cached(model:str) -> bool:
    "True when `model` is in the Hugging Face cache."
    from huggingface_hub.constants import HF_HUB_CACHE
    d = Path(HF_HUB_CACHE)/f"models--{model.replace('/', '--')}"/'snapshots'
    return d.is_dir() and any(d.iterdir())

def cache_bytes(model:str) -> int:
    "The bytes of `model` in the Hugging Face cache so far, with the files that are still downloading."
    from huggingface_hub.constants import HF_HUB_CACHE
    d = Path(HF_HUB_CACHE)/f"models--{model.replace('/', '--')}"/'blobs'
    return sum(f.stat().st_size for f in d.iterdir()) if d.is_dir() else 0

`search_models` looks up mlx-community speech to text models on Hugging Face, with their size. It drops text to speech, aligners
and diarization models, which the `mlx` engine cannot use.

In [ ]:
#| export
STT_WORDS = ('whisper', 'parakeet', 'asr', 'voxtral', 'canary', 'moonshine', 'granite', 'sensevoice', 'nemotron', 'stt', 'speech')
NOT_STT = ('tts', 'aligner', 'diariz', 'music', 'vad', 'speech-tokenizer')

def is_stt(model:str) -> bool:
    "True when the repo name looks like a speech to text model."
    m = model.lower()
    return any(w in m for w in STT_WORDS) and not any(w in m for w in NOT_STT)

def model_size(model:str, api=None) -> int:
    "The bytes of all the files of `model` on Hugging Face."
    from huggingface_hub import HfApi
    return sum(s.size or 0 for s in (api or HfApi()).model_info(model, files_metadata=True).siblings)

TTS_WORDS = ('tts', 'kokoro', 'chatterbox', 'orpheus', 'csm-', 'sesame', 'marvis', 'voxcpm', 'spark', 'soprano', 'dia-', 'outetts', 'kitten',
             'vibevoice', 'fish-audio', 'omnivoice', 'melotts', 'zonos', 'bark')
NOT_TTS = ('asr', 'whisper', 'stt', 'tokenizer', 'codec', 'aligner', 'vad', 'snac', 'encodec', 'mimi')

def is_tts(model:str, tag:str=None) -> bool:
    "True when the repo looks like a text to speech model: its Hugging Face tag `tag` says so, or its name does."
    m = model.lower()
    return not any(w in m for w in NOT_TTS) and (tag == 'text-to-speech' or any(w in m for w in TTS_WORDS))

def search_models(q:str='', limit:int=15, api=None, kind:str='stt') -> list:
    "mlx-community models that match `q`, most downloaded first, with their size: speech to text, or text to speech when `kind` is `tts`."
    from huggingface_hub import HfApi
    api = api or HfApi()
    ok = (lambda m: is_stt(m.id)) if kind == 'stt' else (lambda m: is_tts(m.id, getattr(m, 'pipeline_tag', None)))
    ms = [m for m in api.list_models(author='mlx-community', search=q or None, sort='downloads', limit=400) if ok(m)][:limit]
    with ThreadPoolExecutor(8) as ex: sizes = list(ex.map(lambda m: model_size(m.id, api), ms))
    return [dict(model=m.id, downloads=m.downloads or 0, size=s) for m, s in zip(ms, sizes)]

The filter keeps speech to text and drops the others. `is_tts` is its mirror for the speaking voice: it keeps text to speech and drops speech to text.

In [ ]:
test_eq([is_stt(m) for m in ('mlx-community/Qwen3-ASR-0.6B-8bit', 'mlx-community/whisper-tiny-asr-fp16', 'mlx-community/Qwen3-TTS-12Hz-1.7B-Base-bf16', 'mlx-community/silero-vad')],
        [True, True, False, False])
test_eq([is_tts(m) for m in ('mlx-community/Kokoro-82M-bf16', 'mlx-community/Qwen3-TTS-12Hz-0.6B-Base-8bit', 'mlx-community/whisper-tiny-asr-fp16', 'mlx-community/some-model')],
        [True, True, False, False])
test_eq(is_tts('mlx-community/some-model', 'text-to-speech'), True)

`search_models` with `kind='tts'` gives only the text to speech models. This test uses a fake Hugging Face API, so it does not use the network.

In [ ]:
class _Api:
    def list_models(self, **kw): return [AttrDict(id=i, downloads=n, pipeline_tag=t) for i, n, t in
                                         (('mlx-community/Kokoro-82M-bf16', 9, 'text-to-speech'), ('mlx-community/whisper-tiny-asr-fp16', 8, None), ('mlx-community/odd-voice', 7, 'text-to-speech'))]
    def model_info(self, m, files_metadata=True): return AttrDict(siblings=[AttrDict(size=10)])
test_eq([r['model'] for r in search_models('', api=_Api(), kind='tts')], ['mlx-community/Kokoro-82M-bf16', 'mlx-community/odd-voice'])
test_eq([r['model'] for r in search_models('', api=_Api())], ['mlx-community/whisper-tiny-asr-fp16'])

`fetch` downloads a model and notes it in the settings. `remove` deletes a model from the cache, but only a model that `fetch` downloaded.
Other entries in the cache belong to other programs.

In [ ]:
#| export
def fetch(model:str):
    "Download `model` into the Hugging Face cache; note it when it was not there before."
    from huggingface_hub import snapshot_download
    new = not cached(model)
    snapshot_download(model)
    if new:
        d = settings(); d['downloaded'] = sorted({*d['downloaded'], model}); save_settings(d)

def remove(model:str):
    "Delete `model` from the Hugging Face cache. Only a model that `fetch` downloaded."
    d = settings()
    if model not in d['downloaded']: raise ValueError(f'{model} was not downloaded by abhyasa, so it stays')
    for slot, (r, _) in _slots.items():
        if r.key and r.key[1] == model: unload(slot)
    from huggingface_hub import scan_cache_dir
    info = scan_cache_dir()
    revs = [r.commit_hash for repo in info.repos if repo.repo_id == model for r in repo.revisions]
    if revs: info.delete_revisions(*revs).execute()
    d['downloaded'].remove(model); save_settings(d)

`remove` refuses a model that abhyasa did not download.

In [ ]:
test_fail(lambda: remove(DFLT_GEMMA), contains='stays')

## A natural voice

The board of the screen guide says one line for each step. The browser can say it with its own voice (`speechSynthesis`), but that voice sounds
like a machine. `speak` gives a natural voice. There are three engines:

- `kokoro` runs Kokoro 82M on this Mac with mlx-audio. It is small (390 MB), fast, and it gives the start time of each word.
  Install it with `pip install abhyasa[voice]`.
- `hf` runs any other text to speech model that mlx-audio can load, from Hugging Face: Qwen3-TTS, Chatterbox, CSM, Spark, Soprano and more.
- `openai` sends the line to the OpenAI speech API. It is off until you set a key. Each line passes ramabana's guard first.

`browser` is the fourth choice of the setting `voice.out`, and the default. Then the page speaks, and this module makes no audio.

These are the defaults. The Kokoro voices are the English ones of its Hugging Face repo: `a` is American English, `b` is British English, `f` is a woman, `m` is a man. The OpenAI voices are the voices of its speech API.

In [ ]:
#| export
DFLT_KOKORO = 'mlx-community/Kokoro-82M-bf16'
DFLT_SPEECH = 'gpt-4o-mini-tts'
OUT_ENGINES = ('browser', 'kokoro', 'openai', 'hf')
KOKORO_VOICES = ('af_heart', 'af_bella', 'af_nicole', 'af_sarah', 'af_sky', 'af_nova', 'af_alloy', 'af_aoede', 'af_jessica', 'af_kore', 'af_river',
                 'am_michael', 'am_fenrir', 'am_puck', 'am_echo', 'am_eric', 'am_liam', 'am_onyx', 'am_adam',
                 'bf_emma', 'bf_isabella', 'bf_alice', 'bf_lily', 'bm_george', 'bm_fable', 'bm_lewis', 'bm_daniel')
OPENAI_VOICES = ('marin', 'cedar', 'coral', 'alloy', 'ash', 'ballad', 'echo', 'fable', 'nova', 'onyx', 'sage', 'shimmer', 'verse')
OPENAI_SPEECH = (DFLT_SPEECH, 'tts-1', 'tts-1-hd')
OUT_DFLT = dict(kokoro=(DFLT_KOKORO, 'af_heart'), openai=(DFLT_SPEECH, 'marin'), hf=('', ''))

`out_resolve` fills in the default model and voice of an engine. `out_current` gives the voice of the settings, with its speed. `out_where` says where the voice runs.

In [ ]:
#| export
def out_resolve(engine:str, model:str='', voice:str='') -> tuple:
    "The engine, model and voice for a name; an empty model or voice gets the default of the engine. `browser` has no model."
    if engine not in OUT_ENGINES: raise ValueError(f'unknown voice engine {engine!r}')
    if engine == 'browser': return 'browser', '', ''
    m, v = OUT_DFLT[engine]
    if engine == 'hf' and not model: raise ValueError('Pick a Hugging Face model for the voice.')
    return engine, model or m, voice or v

def out_current() -> tuple:
    "The voice in use: `(engine, model, voice, speed)` from the settings."
    e = val('voice.out')
    try: e, m, v = out_resolve(e, val('voice.out_model'), val('voice.out_voice'))
    except ValueError: e, m, v = 'browser', '', ''   # `hf` with no model yet
    return e, m, v, val('voice.out_speed')

def out_where(engine:str) -> str: return 'hosted' if engine == 'openai' else 'on this Mac'

The settings give the browser voice. Kokoro gets its model and `af_heart`. The `hf` engine needs a model.

In [ ]:
test_eq(out_current(), ('browser', '', '', 1.))
test_eq(out_resolve('kokoro'), ('kokoro', DFLT_KOKORO, 'af_heart'))
test_eq(out_resolve('openai', '', 'coral'), ('openai', DFLT_SPEECH, 'coral'))
test_fail(lambda: out_resolve('hf'), contains='Hugging Face model')
test_fail(lambda: out_resolve('espeak'), contains='unknown')

### The time of each word

The board lights each word when the voice says it. Kokoro gives the start time of each token. For a model that gives no times, `by_letters`
shares the length of the audio among the words by their letters. `word_times` takes the time of the model where it has one, and the
estimate for the other words. A time is never earlier than the time of the word before it.

In [ ]:
#| export
def by_letters(text:str, secs:float) -> list:
    "A start time in seconds for each word of `text` in audio of `secs` seconds: each word takes a share by its letters."
    ws = text.split()
    n, t, out = sum(len(w) + 1 for w in ws) or 1, 0., []
    for w in ws: out.append(round(t, 3)); t += secs * (len(w) + 1) / n
    return out

def word_times(text:str, marks:dict, secs:float) -> list:
    "The start time of each word of `text`: from `marks` (word index -> seconds) when the model gave one, else by its letters; never earlier than the word before."
    out, last = [], 0.
    for k, t in enumerate(by_letters(text, secs)):
        last = max(last, marks.get(k, t)); out.append(round(last, 3))
    return out

Two words of 2 and 6 letters in 1 second: the first word starts at 0, the second after 3/10 of the second. A mark that is too early does not go back in time.

In [ ]:
test_eq(by_letters('is hidden', 1.), [0., .3])
test_eq(word_times('a b c', {1: .5, 2: .1}, 1.5), [0., .5, .5])
test_eq(by_letters('', 1.), [])

### WAV

`to_wav` makes WAV bytes (mono, 16 bit) from the float samples of a model. `clean_wav` writes the header of a WAV again with its true length.
OpenAI streams its WAV, so its header can give a wrong length.

In [ ]:
#| export
def to_wav(samples, rate:int) -> bytes:
    "WAV bytes (mono, 16 bit) for float samples from -1 to 1."
    import io, numpy as np
    a = (np.clip(np.asarray(samples, dtype=np.float32).reshape(-1), -1, 1) * 32767).astype('<i2')
    b = io.BytesIO()
    with wave.open(b, 'wb') as w: w.setnchannels(1); w.setsampwidth(2); w.setframerate(rate); w.writeframes(a.tobytes())
    return b.getvalue()

def clean_wav(audio:bytes) -> bytes:
    "`audio` with a header that gives the true length of its samples."
    import io
    with wave.open(io.BytesIO(audio)) as w: ps, frames = w.getparams(), w.readframes(len(audio))
    b = io.BytesIO()
    with wave.open(b, 'wb') as o: o.setparams(ps._replace(nframes=0)); o.writeframes(frames)
    return b.getvalue()

One second of samples at 24 kHz is one second of WAV. A header that says the data is very long becomes a header with the true length.

In [ ]:
test_eq(wav_seconds(to_wav([0.] * 24000, 24000)), 1.)
import struct
w = to_wav([0.] * 2400, 24000)
bad = w[:4] + struct.pack('<I', 0xFFFFFFFF) + w[8:40] + struct.pack('<I', 0xFFFFFFFF) + w[44:]   # as a stream: the RIFF and data sizes are unknown
test_eq(wav_seconds(clean_wav(bad)), .1)

### Kokoro and the other mlx-audio models

`_tts_load` loads a model with the generic loader of mlx-audio. It can load each text to speech model of mlx-audio, from a Hugging Face repo id.
The model stays in the `tts` slot of `resident`.

Kokoro reads English text with misaki, and misaki uses the small English model of spaCy. spaCy installs that model with pip, which a uv
environment does not have. Thus `spacy_ready` gets it from its Hugging Face repo (`spacy/en_core_web_sm`) one time, and unpacks it into
`cfg_dir()/'site'`. Its model is 3.7, and spaCy 3.8 reads it well, so `spacy_ready` hides the warning about the version.

In [ ]:
#| export
SPACY_EN = 'en_core_web_sm'

def spacy_ready():
    "Make the small English spaCy model importable: from its Hugging Face repo, unpacked one time into `cfg_dir()/'site'`."
    import sys, zipfile, importlib, warnings, spacy
    warnings.filterwarnings('ignore', message=r'\[W095\]')   # the repo has model 3.7: spaCy 3.8 reads it well, so the warning is noise
    if spacy.util.is_package(SPACY_EN): return
    d = cfg_dir()/'site'
    if not (d/SPACY_EN).is_dir():
        from huggingface_hub import hf_hub_download
        with zipfile.ZipFile(hf_hub_download(f'spacy/{SPACY_EN}', f'{SPACY_EN}-any-py3-none-any.whl')) as z: z.extractall(d)
    if str(d) not in sys.path: sys.path.append(str(d)); importlib.invalidate_caches()

def _tts_load(model):
    from mlx_audio.tts.utils import load
    return load(model)

Kokoro gives a start time for each token of its text, from the length that it gives each sound. `kokoro_marks` finds each token
in the line, and gives the word of the line where the token starts its time. The time of a part of the audio adds the length of the parts before it.

In [ ]:
#| export
def kokoro_marks(text:str, results, rate:int=24000) -> tuple:
    "The audio parts and the word marks (word index -> seconds) from the results of a Kokoro pipeline for `text`."
    import bisect, numpy as np
    starts = [m.start() for m in re.finditer(r'\S+', text)]
    parts, marks, t, pos = [], {}, 0., 0
    for r in results:
        if r.audio is None: continue
        for tk in r.tokens or []:
            i = text.find(tk.text, pos) if tk.text else -1
            if i < 0: continue
            pos = i + len(tk.text)
            if getattr(tk, 'start_ts', None) is not None: marks.setdefault(bisect.bisect_right(starts, i) - 1, round(t + tk.start_ts, 3))
        a = np.asarray(r.audio, dtype=np.float32).reshape(-1)
        parts.append(a); t += len(a) / rate
    return parts, marks

`_mlx_say` speaks with Kokoro through its pipeline, for the times. Another model speaks through its `generate`, with the voice and the speed when it takes them.

In [ ]:
#| export
def _mlx_say(text, model, voice, speed, instructions=''):
    import numpy as np
    m = resident(('tts', model), lambda: _tts_load(model), slot='tts')
    rate = getattr(m, 'sample_rate', 24000)
    if hasattr(m, '_get_pipeline'):   # Kokoro: its pipeline gives the start time of each token
        spacy_ready()
        lang = voice[0] if voice and voice[0] in 'abefhijpz' else 'a'
        parts, marks = kokoro_marks(text, m._get_pipeline(lang)(text, voice=voice or 'af_heart', speed=speed, split_pattern=None), rate)
    else:
        ps = inspect.signature(m.generate).parameters
        rs = list(m.generate(text, **{k: v for k, v in dict(voice=voice, speed=speed).items() if v and k in ps}))
        parts, marks, rate = [np.asarray(r.audio, dtype=np.float32).reshape(-1) for r in rs], {}, (rs[0].sample_rate if rs else rate)
    return to_wav(np.concatenate(parts) if parts else np.zeros(0), rate), marks

### OpenAI

The `openai` engine posts the line to the OpenAI speech API and asks for WAV. `instructions` can set the tone ("calm, like a teacher"); only
the `gpt-4o` models read it. The owner's rule is that each text that goes to a hosted model passes ramabana's guard
(`abhyasa.guide.guard`, with `ramabana.guard.outgoing`). It masks a secret and refuses personal data when the privacy setting says so.

In [ ]:
#| export
OPENAI_SPEECH_URL = 'https://api.openai.com/v1/audio/speech'
OPENAI_SPEC = AttrDict(name='openai', local=False)   # the route for the guard: a model off this Mac

def _openai_say(text, model, voice, speed, instructions=''):
    key, _ = openai_key()
    if not key: raise RuntimeError('There is no OpenAI key. Set one on the Settings page.')
    from abhyasa.guide import guard
    body = dict(model=model, voice=voice, input=guard(text, OPENAI_SPEC), response_format='wav', speed=speed)
    if instructions and not model.startswith('tts-1'): body['instructions'] = guard(instructions, OPENAI_SPEC)
    r = (http_post or __import__('httpx').post)(OPENAI_SPEECH_URL, headers={'Authorization': f'Bearer {key}'}, json=body, timeout=60)
    r.raise_for_status()
    return clean_wav(r.content), {}

SAY = dict(kokoro=_mlx_say, hf=_mlx_say, openai=_openai_say)

### speak

`speak` gives the WAV bytes of one line and the start time of each of its words. Without an engine, it uses the voice of the settings.
For the browser voice, it refuses: the page speaks then.

In [ ]:
#| export
def speak(text:str, engine:str=None, model:str=None, voice:str=None, speed:float=None, instructions:str='') -> tuple:
    "WAV bytes for `text`, and the start time in seconds of each of its words (split on white space)."
    e, m, v, s = out_current()
    if engine: e, m, v = out_resolve(engine, model or '', voice or '')
    if e == 'browser': raise ValueError('The browser speaks with its own voice. There is no audio here.')
    wav, marks = SAY[e](text, model or m, voice or v, float(speed or s), instructions)
    return wav, word_times(text, marks, wav_seconds(wav))

### Tests with a fake model

A fake Kokoro: its pipeline gives one result for each word, of 0.25 s each, with a token that starts 0.05 s into its part.
The loader is a fake too, so the test downloads nothing. `spacy_ready` is a fake as well.

In [ ]:
import numpy as np
class _Tk(AttrDict): pass
class _Kokoro:
    sample_rate = 24000
    def __init__(self): self.said = []
    def _get_pipeline(self, lang):
        def run(text, voice, speed, split_pattern):
            self.said.append((lang, text, voice, speed))
            for w in text.split(): yield AttrDict(audio=np.zeros((1, 6000), np.float32), tokens=[_Tk(text=w.strip('.,'), start_ts=.05)])
        return run
loads = []
_tts_load = lambda m: loads.append(m) or _Kokoro()
spacy_ready, _spacy = (lambda: None), spacy_ready

`speak` gives one second of WAV for four words, and a time for each word that goes up. The second call does not load the model again.

In [ ]:
wav, words = speak('Noise goes in, slowly.', 'kokoro')
test_eq(wav_seconds(wav), 1.)
test_eq(words, [.05, .3, .55, .8])
test_eq(words, sorted(words))
speak('Again.', 'kokoro', voice='bm_george', speed=1.2)
test_eq(loads, [DFLT_KOKORO])
test_eq(_slots['tts'][0].obj.said[-1], ('b', 'Again.', 'bm_george', 1.2))

A model that is not Kokoro speaks through `generate`. It gives no times, so the times come from the letters. It gets the voice only when it takes one.

In [ ]:
class _Plain:
    sample_rate = 16000
    def generate(self, text, voice=None):
        self.voice = voice
        yield AttrDict(audio=np.zeros(16000, np.float32), sample_rate=16000)
_tts_load = lambda m: _Plain()
wav, words = speak('one two three', 'hf', 'mlx-community/plain-tts', voice='calm')
test_eq((wav_seconds(wav), words), (1., by_letters('one two three', 1.)))
test_eq(_slots['tts'][0].obj.voice, 'calm')
unload('tts')

The settings choose the voice when `speak` has no engine. The browser voice makes no audio here.

In [ ]:
test_fail(lambda: speak('hello'), contains='browser')
update_settings('voice', out='kokoro', out_voice='af_bella')
_tts_load = lambda m: _Kokoro()
speak('hello')
test_eq(_slots['tts'][0].obj.said[-1][2], 'af_bella')
update_settings('voice', out='browser', out_voice='')

The OpenAI engine refuses with no key. With a key, it sends the text after the guard, with the model, the voice and the tone. Here a fake guard puts a mark on each text, so the test sees that each text passed it.

In [ ]:
import abhyasa.guide
real_guard, abhyasa.guide.guard = abhyasa.guide.guard, lambda s, model=None, mode=None: f'[{model.name}] {s}'
test_fail(lambda: speak('hello', 'openai'), contains='no OpenAI key')
set_openai_key('sk-test-1234')
class _W:
    content = to_wav([0.] * 12000, 24000)
    def raise_for_status(self): pass
sent = {}
http_post = lambda url, **kw: sent.update(url=url, **kw) or _W()
wav, words = speak('Look at the noise.', 'openai', instructions='calm, like a teacher')
test_eq(sent['url'], OPENAI_SPEECH_URL)
test_eq(sent['json'], dict(model=DFLT_SPEECH, voice='marin', input='[openai] Look at the noise.', response_format='wav', speed=1., instructions='[openai] calm, like a teacher'))
test_eq((wav_seconds(wav), len(words)), (.5, 4))

`tts-1` does not read `instructions`, so the engine does not send them. Then the test puts back the real guard, the real HTTP client and the real loaders.

In [ ]:
speak('Hi.', 'openai', 'tts-1', instructions='calm')
assert 'instructions' not in sent['json']
abhyasa.guide.guard, http_post = real_guard, None
set_openai_key('')
_tts_load, spacy_ready = (lambda m: __import__('mlx_audio.tts.utils', fromlist=['load']).load(m)), _spacy

### One worker for the voice

`Speaker` has one worker thread for all the calls of the speaking voice, separate from the speech to text worker. Thus a line of the board
does not wait for a segment of dictation. It keeps the lines that it said in this run (at most `most`), by engine, model, voice, speed and
text. The board asks for the same line again when the learner goes back a step, and that line then comes at once.
Two requests for the same line share one call. `warm` loads the model in use with one short line, so the first line of a board does not
wait for the load. The browser voice and OpenAI have nothing to load. A model that is not on this Mac yet is not downloaded for a line:
it can be large, and the Settings page shows its download.

In [ ]:
#| export
class Speaker:
    "One worker thread for the speaking voice, and the lines that it said in this run."
    def __init__(self, fn=None, most:int=64, here=None):
        self.fn, self.most, self.lines, self.status, self.warmed = fn or speak, most, {}, 'not loaded', None
        self.here = here or (lambda m: cached(m))   # a model that is not on this Mac is not downloaded here: the Settings page gets it
        self.ex, self.lock = ThreadPoolExecutor(1, thread_name_prefix='voice'), threading.Lock()

    def ready(self, engine:str, model:str):
        "Raise when the model of an mlx engine is not on this Mac yet."
        if engine in ('kokoro', 'hf') and not self.here(model): raise ValueError(f'The voice model {model} is not on this Mac. Download it on the Settings page.')

    def key(self, text:str, engine:str=None, model:str=None, voice:str=None, speed:float=None) -> tuple:
        "The key of a line: `(engine, model, voice, speed, text)`, with the settings for what is not given."
        e, m, v, s = out_current()
        if engine: e, m, v = out_resolve(engine, model or '', voice or '')
        return e, model or m, voice or v, float(speed or s), ' '.join(str(text or '').split())

    def say(self, text:str, **kw) -> dict:
        "The audio of one line: `dict(wav, words, engine, model, voice, speed, where, cached)`."
        k = self.key(text, **kw)
        if k[0] == 'browser': raise ValueError('The browser speaks with its own voice. There is no audio here.')
        if not k[4]: raise ValueError('There is no text to say.')
        self.ready(*k[:2])
        with self.lock:
            hit = k in self.lines
            f = self.lines.pop(k) if hit else self.ex.submit(self.fn, k[4], *k[:4])
            self.lines[k] = f   # the newest at the end
            while len(self.lines) > self.most: self.lines.pop(next(iter(self.lines)))
        try: wav, words = f.result()
        except Exception:
            with self.lock: self.lines.pop(k, None)   # a failed line is not kept, so the next request tries again
            raise
        return dict(wav=wav, words=words, engine=k[0], model=k[1], voice=k[2], speed=k[3], where=out_where(k[0]), cached=hit)

    def warm(self):
        "Load the voice in use on the worker with one short line; None when there is nothing to load."
        e, m, v, s = out_current()
        if e in ('browser', 'openai'): self.status = 'ready'; return None
        if self.warmed == (e, m) and self.status != 'loading' or not self.here(m): return None
        self.status, self.warmed = 'loading', (e, m)
        def go():
            try: self.fn('Ready.', e, m, v, s); self.status = 'ready'
            except Exception as ex: self.status, self.warmed = f'failed: {ex}', None
        return self.ex.submit(go)

A fake voice counts its calls. The same line comes from the cache, and a new speed is a new line.

In [ ]:
calls = []
def fake_speak(text, engine, model, voice, speed):
    calls.append((text, engine, voice, speed)); time.sleep(.02)
    return to_wav([0.] * 2400, 24000), by_letters(text, .1)
sk = Speaker(fake_speak, most=2, here=lambda m: True)
test_fail(lambda: sk.say('hi'), contains='browser')
r = sk.say('Look  here.', engine='kokoro')
test_eq((r['cached'], r['where'], r['voice'], r['words']), (False, 'on this Mac', 'af_heart', by_letters('Look here.', .1)))
test_eq(sk.say('Look here.', engine='kokoro')['cached'], True)
sk.say('Look here.', engine='kokoro', speed=1.5)
test_eq(len(calls), 2)

Two requests at the same time for a new line make one call. The cache keeps at most `most` lines: the oldest goes first.

In [ ]:
with ThreadPoolExecutor(2) as ex: list(ex.map(lambda _: sk.say('Both at once.', engine='kokoro'), range(2)))
test_eq(len(calls), 3)
test_eq(len(sk.lines), 2)
test_eq(sk.say('Look here.', engine='kokoro')['cached'], False)

A failed line is not kept. `warm` loads the voice of the settings one time, and does nothing for the browser voice. A model that is not here refuses.

In [ ]:
sk = Speaker(lambda *a: 1 / 0, here=lambda m: True)
test_fail(lambda: sk.say('x', engine='kokoro'))
test_eq(sk.lines, {})
test_is(Speaker(fake_speak).warm(), None)
update_settings('voice', out='kokoro')
sk = Speaker(fake_speak, here=lambda m: True); calls.clear()
sk.warm().result(); test_is(sk.warm(), None)
test_eq((sk.status, calls), ('ready', [('Ready.', 'kokoro', 'af_heart', 1.)]))
update_settings('voice', out='browser')
sk = Speaker(fake_speak, here=lambda m: False)
test_fail(lambda: sk.say('x', engine='kokoro'), contains='not on this Mac')

## Segments in order

The browser sends one segment each time you stop to think. `Speech` has one worker thread for all the model calls.
Thus only one model is in memory, and the segments run one at a time, in the order they came.
A `Dictation` collects the segments of one answer. `next` gives the text in the order of the segment numbers,
also when a slow request arrives late. When the browser sends a segment again (after an error), the code does not use it two times.

In [ ]:
#| export
class Dictation:
    "The segments of one spoken answer: each is transcribed on the speech worker, and the text comes out in order."
    def __init__(self, speech, hint=''):
        store_attr()
        self.seen, self.ready, self.out, self.n_done, self.total, self.t = set(), {}, 0, 0, None, time.time()
        self.cv = threading.Condition()

    def add(self, seq:int, audio:bytes) -> bool:
        "Queue segment `seq`; False when it came before."
        with self.cv:
            if seq in self.seen: return False
            self.seen.add(seq)
        self.speech.submit(self._run, seq, audio)
        return True

    def _run(self, seq, audio):
        try: r = dict(seq=seq, text=self.speech.fn(audio, hint=self.hint).strip())
        except Exception as e: r = dict(seq=seq, text='', error=str(e))
        with self.cv: self.ready[seq] = r; self.n_done += 1; self.cv.notify_all()

    def stop(self, n:int):
        "There are `n` segments in all."
        with self.cv: self.total = n; self.cv.notify_all()

    @property
    def pending(self) -> int: return len(self.seen) - self.n_done

    def next(self, timeout=None):
        "The next result in order; `dict(done=True)` after the last one; None at the timeout."
        with self.cv:
            if not self.cv.wait_for(lambda: self.out in self.ready or self.total == self.out, timeout): return None
            if self.out not in self.ready: return dict(done=True)
            self.out += 1
            return self.ready.pop(self.out - 1)

`Speech` owns the worker. `warm` loads the model in use with half a second of silence, so the first segment does not wait for it.
`use` changes the engine and model, saves the change, and warms the new model.

In [ ]:
#| export
class Speech:
    "One worker thread for every speech model call, and the open dictations."
    def __init__(self, fn=None):
        self.fn, self.runs, self.status = fn or transcribe, {}, 'not loaded'
        self.ex = ThreadPoolExecutor(1, thread_name_prefix='speech')

    def submit(self, f, *a, **kw): return self.ex.submit(f, *a, **kw)

    def warm(self):
        "Load the model in use on the worker."
        def go():
            try: self.fn(silence(), hint=''); self.status = 'ready'
            except Exception as ex: self.status = f'failed: {ex}'
        self.status = 'loading'   # at once, so a page drawn now shows it
        return self.submit(go)

    def use(self, engine:str, model:str=''):
        "Switch to `engine` and `model`, save the choice, and warm the new model."
        engine, model = resolve(engine, model)
        update_settings('stt', engine=engine, model=model)
        return self.warm()

    def start(self, hint:str='') -> str:
        "Open a dictation; returns its id. Dictations older than an hour are dropped."
        for k in [k for k, d in self.runs.items() if time.time() - d.t > 3600]: self.runs.pop(k, None)
        did = uuid.uuid4().hex[:10]
        self.runs[did] = Dictation(self, hint)
        return did

### Tests with a fake engine

This fake engine is slow on even segments, so the requests finish out of order. The text must still come out in order.

In [ ]:
calls = []
def slow_fake(audio, hint=''):
    n = int(audio.decode())
    calls.append(n)
    time.sleep(.05 if n % 2 == 0 else 0)
    return f'segment {n} ({hint})'
sp = Speech(slow_fake)
d = sp.runs[sp.start('hetu')]

Segments 0 to 5 arrive out of order, from several threads. Segment 3 comes two times.

In [ ]:
order = [2, 0, 1, 3, 3, 5, 4]
with ThreadPoolExecutor(4) as ex: added = list(ex.map(lambda n: d.add(n, str(n).encode()), order))
test_eq(sum(added), 6)
d.stop(6)
out = []
while not (r := d.next(5)).get('done'): out.append(r)
test_eq([r['seq'] for r in out], list(range(6)))
test_eq(out[0]['text'], 'segment 0 (hetu)')
test_eq(sorted(calls), list(range(6)))

All the model calls ran on the one worker thread, one at a time.

In [ ]:
seen = set()
sp.submit(lambda: seen.add(threading.current_thread().name)).result()
test_eq(len(seen), 1)
test_eq(d.pending, 0)

An engine error gives an empty text with the error. It does not stop the segments after it.

In [ ]:
def flaky(audio, hint=''):
    if audio == b'bad': raise RuntimeError('no model')
    return 'ok'
sp = Speech(flaky)
d = sp.runs[sp.start()]
d.add(0, b'bad'); d.add(1, b'good'); d.stop(2)
test_eq(d.next(5), dict(seq=0, text='', error='no model'))
test_eq(d.next(5), dict(seq=1, text='ok'))
test_eq(d.next(5), dict(done=True))

Without `stop`, `next` waits and then gives None. `use` saves the choice and warms the new model with silence.

In [ ]:
d = sp.runs[sp.start()]
test_is(d.next(.05), None)
heard = []
sp = Speech(lambda a, hint='': heard.append(wav_seconds(a)) or '')
sp.use('parakeet').result()
test_eq(current(), ('mlx', DFLT_PARAKEET))
test_eq(heard, [.5])
assert sp.status.startswith('ready'), sp.status
update_settings('stt', engine='gemma', model='')

## Real models

These tests use a real clip and real models. They are slow. Run them with `nbdev-test --flags slow`.

In [ ]:
#| slow
clip = first(p for p in (Path('../../rishi/nbs/speech.wav'), Path('../../../rishi/nbs/speech.wav')) if p.exists())   # from the repo or a worktree
if clip:
    for e in ('gemma', 'parakeet', 'mlx:mlx-community/Qwen3-ASR-0.6B-8bit'): print(e, transcribe(clip, e, hint='Tennyson'))
    unload()

In [ ]:
#| slow
t = time.time(); wav, words = speak('The forward process adds a little noise at each step.', 'kokoro')
print(f'Kokoro: {time.time() - t:.1f} s cold, {wav_seconds(wav):.1f} s of audio, words at {words}')
Path('/tmp/abhyasa-kokoro.wav').write_bytes(wav)
unload('tts')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()